# 16 · Multi-variable forecast verification & ERA5 bias correction

Evaluates all 9 archived Open-Meteo forecast models (`ecmwf_ifs`, `gfs`, `icon`, `gem`, `ecmwf_aifs`, `icon_eu`, `icon_d2`, `arome`, `hrdps`) against the calibrated IGC atmospheric truth tables built by notebook 15 (`database/truth/`) across four physical pillars — without re-downloading wind:

1. **Pillar 1 — Clean wind (0 API calls):** re-scores the existing wind cache (`database/weather/hist_forecast/` from notebook 14) against `truth_wind_groups.parquet` and `truth_glide_wind.parquet`.
2. **Pillar 2 — Top of lift ASL (`truth_ceiling.parquet`):** verifies native BLH (`gfs`, `ecmwf_ifs` 9 km) vs dry-adiabatic `parcel` BLH and calibrates the optimal convective trigger excess (`+0.0..+2.5 °C`) in 5-fold competition-grouped CV.
3. **Pillar 3 — Core 60 s climb strength (`truth_climb_groups.parquet`):** tests which forecast model's thermodynamics best predict real `core60_p75_ms` (the #1 physical driver of `winner_kmh`, `r = +0.556`) in 5-fold competition-grouped CV.
4. **Pillar 4 — End-to-end task speed & goal share (`truth_tasks.parquet`, 2022+):** runs leakage-free `task_v1` folds on each model's historical route weather — comparing **raw** vs **thermo-only quantile mapping** vs **full-weather quantile mapping** to ERA5, and **equal** vs **clean-wind weighted** vs **multi-variable composite** blends.

Writes `database/weather/hist_forecast_thermo/` and `database/models/forecast_multivar_eval/` (`forecast_calibration.json`, `forecast_multivar_weights.json`, and CSV/JSON scorecards). Does **not** overwrite live `web_data/forecast_weights.json` until CV gains are verified.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
BRANCH = "main"             # set to "feat/forecast-multivar-eval" if running before merge
REPO = "/content/Pre-task-analyzer"

FROM_YEAR = 2022            # earliest competition year to verify
PAUSE_S = 0.4               # delay between Open-Meteo requests (s)
MAX_SITES = None            # set to e.g. 20 for a quick smoke test; None runs every site
CENTROID_CHECK_N = 0        # optional: set to e.g. 100 to test regional models at real thermal centroid vs chip centre

## 2. Mount Drive, get the code

In [ ]:
import json, os, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import auth, drive
from googleapiclient.discovery import build

auth.authenticate_user()
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):
        return path_up(folder_id, "/content/drive/MyDrive")
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut in Drive first.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
THERMO_DIR = os.path.join(DB, "weather", "hist_forecast_thermo")
OUT = os.path.join(DB, "models", "forecast_multivar_eval")
assert os.path.isfile(os.path.join(DB, "truth", "truth_wind_groups.parquet")), "Run notebook 15 first: database/truth/ is missing"
subprocess.run(["pip", "install", "-q", "lightgbm"], check=True)
print("Database in", DB)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO], check=True)


def run(*cmd):
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="")
        code = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("Stopped.")
        raise
    if code != 0:
        raise RuntimeError(f"{os.path.basename(cmd[0])} stopped with exit code {code}")
    return code


def download_thermo(model_id):
    """Download or resume archived thermodynamic variables for one model into hist_forecast_thermo/."""
    cmd = [
        f"{REPO}/scripts/forecast_multivar_eval.py",
        DB,
        "--from-year", str(FROM_YEAR),
        "--models", model_id,
        "--pause", str(PAUSE_S),
        "--download-only",
    ]
    if MAX_SITES is not None:
        cmd += ["--max-sites", str(MAX_SITES)]
    return run(*cmd)

## 3. Pillar 1 immediate check: re-score cached winds against clean IGC truth (0 API calls)

Runs completely offline against `database/weather/hist_forecast/` (from notebook 14) and `database/truth/truth_wind_groups.parquet` + `truth_glide_wind.parquet` (from notebook 15) to see immediately how model rankings and speed biases change with phase-free thermal wind.

In [ ]:
cmd = [f"{REPO}/scripts/forecast_multivar_eval.py", DB, "--from-year", str(FROM_YEAR), "--wind-only"]
if MAX_SITES is not None:
    cmd += ["--max-sites", str(MAX_SITES)]
run(*cmd)

ws = pd.read_csv(os.path.join(OUT, "wind_clean_scores.csv"))
fm = ws[ws["band"] == "free+middle"].sort_values("rel_rmse_vs_era5").reset_index(drop=True)
display(fm[["model", "groups", "vector_rmse_ms", "old_vector_rmse_ms", "speed_bias_ms", "old_speed_bias_ms", "era5_rmse_ms", "rel_rmse_vs_era5", "dir_error_deg", "dir_within_30"]].round(3))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.5))
ax = axes[0]
y = np.arange(len(fm))
ax.barh(y - 0.18, fm["old_speed_bias_ms"], height=0.35, color="#b5b3aa", label="Old endpoint wind − model")
ax.barh(y + 0.18, fm["speed_bias_ms"], height=0.35, color="#2c9f74", label="Clean (hodo+int) − model")
ax.axvline(0, color="#333", lw=0.9, ls=":")
ax.set_yticks(y)
ax.set_yticklabels(fm["name"])
ax.invert_yaxis()
ax.set_xlabel("Median speed bias (m/s)")
ax.set_title("Speed bias shrinkage across all 9 forecast models")
ax.legend(frameon=False, fontsize=8.5)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
wcv = json.loads(open(os.path.join(OUT, "wind_clean_cv.json"), encoding="utf-8").read())
gl = wcv.get("vs_glide_wind") or {}
if gl:
    g_items = sorted([(k, v) for k, v in gl.items() if v.get("groups")], key=lambda kv: kv[1]["vector_rmse_ms"])
    yg = np.arange(len(g_items))
    colors = ["#9e9d93" if k == "era5" else "#2a78d6" for k, _ in g_items]
    ax.barh(yg, [v["vector_rmse_ms"] for _, v in g_items], color=colors, alpha=0.88)
    for yi, (k, v) in enumerate(g_items):
        ax.text(v["vector_rmse_ms"] + 0.03, yi, f"{v['vector_rmse_ms']:.2f} (bias {v['speed_bias_ms']:+.2f})", va="center", fontsize=8.5)
    ax.set_yticks(yg)
    ax.set_yticklabels([k for k, _ in g_items])
    ax.invert_yaxis()
    ax.set_xlabel("Vector RMSE vs independent multi-heading glide wind (m/s)")
    ax.set_title("Independent cross-check vs glide wind")
    ax.set_xlim(0, max(v["vector_rmse_ms"] for _, v in g_items) * 1.32)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Download archived thermodynamic forecasts (one cell per model)

Downloads only the thermodynamic variables each model publishes (no wind re-downloads!) for the competition chip-hours in `truth_ceiling`, `truth_climb_groups`, and `2022+` task routes. Run the status table below first, then pace individual model cells across Open-Meteo's hourly/daily limits.

In [ ]:
ALL_MODELS = [
    ("ecmwf_ifs", "global"),
    ("gfs", "global"),
    ("icon", "global"),
    ("gem", "global"),
    ("ecmwf_aifs", "global"),
    ("icon_eu", "regional"),
    ("icon_d2", "regional"),
    ("arome", "regional"),
    ("hrdps", "regional"),
]
rows = []
for mid, scope in ALL_MODELS:
    pq_p = os.path.join(THERMO_DIR, f"{mid}.parquet")
    done_p = os.path.join(THERMO_DIR, f"{mid}_fetched.json")
    n_rows = len(pd.read_parquet(pq_p, columns=["chip_id"])) if os.path.isfile(pq_p) else 0
    n_done = len(json.loads(open(done_p, encoding="utf-8").read())) if os.path.isfile(done_p) else 0
    extra = ""
    if mid == "ecmwf_ifs":
        bp = os.path.join(THERMO_DIR, "ecmwf_ifs_blh9k.parquet")
        extra = f"blh9k={len(pd.read_parquet(bp, columns=['chip_id'])):,}" if os.path.isfile(bp) else "blh9k=0"
    rows.append({"model": mid, "scope": scope, "cached_chip_hours": n_rows, "fetched_keys": n_done, "notes": extra})
display(pd.DataFrame(rows))

### 4.1 Global · ECMWF IFS (`ecmwf_ifs025` + `ecmwf_ifs` 9 km native BLH)

In [ ]:
download_thermo("ecmwf_ifs")

### 4.2 Global · GFS (`gfs_global` — native BLH, CAPE, CIN, freezing level)

In [ ]:
download_thermo("gfs")

### 4.3 Global · ICON (`icon_global`)

In [ ]:
download_thermo("icon")

### 4.4 Global · GEM (`gem_global`)

In [ ]:
download_thermo("gem")

### 4.5 Global · ECMWF AIFS (`ecmwf_aifs025_single`)

In [ ]:
download_thermo("ecmwf_aifs")

### 4.6 Regional (Europe) · ICON-EU (`icon_eu`)

In [ ]:
download_thermo("icon_eu")

### 4.7 Regional (Alps / Central Europe) · ICON-D2 (`icon_d2`)

In [ ]:
download_thermo("icon_d2")

### 4.8 Regional (France / Western Alps) · AROME (`meteofrance_arome_france`)

In [ ]:
download_thermo("arome")

### 4.9 Regional (North America) · HRDPS (`gem_hrdps_continental`)

In [ ]:
download_thermo("hrdps")

## 5. Run full offline 4-pillar verification & 5-fold CV (`--skip-download`)

Evaluates all models cached so far across Pillars 1–4, tunes `_parcel_blh_asl` convective trigger excess per model, validates raw vs thermo-only QM vs full-weather QM to ERA5 on held-out competitions, and exports `forecast_calibration.json` and `forecast_multivar_weights.json`.

In [ ]:
cmd = [
    f"{REPO}/scripts/forecast_multivar_eval.py",
    DB,
    "--from-year", str(FROM_YEAR),
    "--models", "all",
    "--skip-download",
]
if MAX_SITES is not None:
    cmd += ["--max-sites", str(MAX_SITES)]
if CENTROID_CHECK_N > 0:
    cmd += ["--regional-centroid-check", str(CENTROID_CHECK_N)]
run(*cmd)

## 6. Pillar 2 · Top of Lift ASL & Parcel BLH trigger calibration

In [ ]:
cs = pd.read_csv(os.path.join(OUT, "ceiling_scores.csv"))
trig = json.loads(open(os.path.join(OUT, "parcel_trigger_cv.json"), encoding="utf-8").read())

best_c = cs[cs["variant"].isin(["native_min_blh_lcl", "parcel_cv_tuned", "parcel_default_1.0C"])].sort_values(["model", "rmse_m"])
display(best_c[["model", "variant", "groups", "rmse_m", "bias_pred_minus_obs_m", "corr_all", "within_site_day_corr", "era5_rmse_m", "rel_rmse_vs_era5"]].round(3))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))
# Panel A: best variant per model vs ERA5 parity
ax = axes[0]
top_c = cs[cs["variant"].isin(["native_min_blh_lcl", "parcel_cv_tuned"])].sort_values("rel_rmse_vs_era5").drop_duplicates("model").sort_values("rel_rmse_vs_era5", ascending=False)
if len(top_c):
    cols_c = ["#2c9f74" if "native" in v else "#2a78d6" for v in top_c["variant"]]
    bars = ax.barh(top_c["name"], top_c["rel_rmse_vs_era5"], color=cols_c, alpha=0.88)
    ax.axvline(1.0, color="#666", ls="--", lw=1.1, label="ERA5 parity (1.00)")
    for b, (_, r) in zip(bars, top_c.iterrows()):
        tag = "native" if "native" in r["variant"] else f"parcel +{trig[r['model']]['recommended_excess_c']:.1f}°C"
        ax.text(b.get_width() + 0.01, b.get_y() + b.get_height() / 2, f"{b.get_width():.3f} ({tag}, day r={r['within_site_day_corr']})", va="center", fontsize=8)
    ax.set_xlim(0.7, max(1.3, float(top_c["rel_rmse_vs_era5"].max()) + 0.35))
    ax.set_xlabel("ASL Top-of-Lift RMSE relative to ERA5 (lower is better)")
    ax.set_title("Top-of-lift ASL skill (green=native BLH, blue=parcel BLH)")
    ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)

# Panel B: parcel excess_c sensitivity curves
ax = axes[1]
for mid in ["ecmwf_ifs", "gfs", "icon", "gem", "ecmwf_aifs", "icon_eu", "icon_d2"]:
    if mid not in trig:
        continue
    gdict = trig[mid].get("grid_eff_top") or {}
    if not gdict:
        continue
    xs = [float(k) for k in sorted(gdict, key=float)]
    ys = [gdict[f"{x:.1f}"]["rmse_m"] for x in xs]
    ax.plot(xs, ys, "o-", label=f"{mid} (opt +{trig[mid]['recommended_excess_c']:.1f}°C)")
ax.axvline(1.0, color="#666", ls=":", lw=1.0, label="Current API default (+1.0°C)")
ax.set_xlabel("Parcel convective trigger excess (°C)")
ax.set_ylabel("ASL Top-of-Lift RMSE vs pilots (m)")
ax.set_title("Parcel BLH trigger tuning")
ax.legend(frameon=False, fontsize=8, ncol=2)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 7. Pillar 3 · Core 60 s climbing speed skill (`core60_p75_ms`)

In [ ]:
cl_df = pd.read_csv(os.path.join(OUT, "climb_scores.csv")).sort_values("rel_rmse_vs_era5")
display(cl_df[["model", "name", "groups", "rmse_ms", "mae_ms", "r2", "corr", "wstar_corr", "era5_rmse_ms", "rel_rmse_vs_era5"]].round(4))

if len(cl_df):
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    p = cl_df.sort_values("rel_rmse_vs_era5", ascending=False)
    cols = ["#2a78d6" if r == "global" else "#2c9f74" for r in p["region"]]
    bars = ax.barh(p["name"], p["rel_rmse_vs_era5"], color=cols, alpha=0.88)
    ax.axvline(1.0, color="#666", ls="--", lw=1.1, label="ERA5 parity (1.00)")
    for b, (_, r) in zip(bars, p.iterrows()):
        ax.text(b.get_width() + 0.006, b.get_y() + b.get_height() / 2, f"{b.get_width():.3f}  (R²={r['r2']:.3f}, w* r={r['wstar_corr']})", va="center", fontsize=8.5)
    ax.set_xlim(0.85, max(1.15, float(p["rel_rmse_vs_era5"].max()) + 0.18))
    ax.set_xlabel("Out-of-fold core60_p75 RMSE relative to ERA5 (lower is better)")
    ax.set_title("Thermodynamic skill predicting IGC core 60 s climb strength (5-fold CV)")
    ax.legend(frameon=False, loc="lower right")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

## 8. Pillar 4 · End-to-end `task_v1`, Quantile Mapping & Multi-Pillar Composite Weights

In [ ]:
e2e = pd.read_csv(os.path.join(OUT, "task_e2e_scores.csv"))
bcv = json.loads(open(os.path.join(OUT, "bias_correction_cv.json"), encoding="utf-8").read())
cal = json.loads(open(os.path.join(OUT, "forecast_calibration.json"), encoding="utf-8").read())
mw = json.loads(open(os.path.join(OUT, "forecast_multivar_weights.json"), encoding="utf-8").read())

print(f"Recommended QM mode: {cal.get('recommended_qm_mode')} (apply={cal.get('recommended_apply')})")
display(e2e.sort_values(["mode", "winner_time_mae_min"]))

# Summary table of blend strategies across raw / thermo_qm / full_qm
b_rows = []
for mode, strats in bcv.items():
    for sname, met in strats.items():
        if met.get("tasks_time"):
            b_rows.append({"mode": mode, "strategy": sname, **met})
bdf = pd.DataFrame(b_rows)
if len(bdf):
    display(bdf[["mode", "strategy", "tasks_time", "winner_time_mae_min", "winner_time_ci95", "within_15_min", "goal_share_mae_pts"]])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
# Panel A: Blend winner time MAE (raw vs thermo_qm vs full_qm)
ax = axes[0]
if len(bdf):
    strats_ord = [s for s in ["equal_mean", "median", "wind_weighted", "multivar_weighted", "era5_reanalysis"] if s in set(bdf["strategy"])]
    modes_ord = [m for m in ["raw", "thermo_qm", "full_qm"] if m in set(bdf["mode"])]
    x = np.arange(len(strats_ord))
    w = 0.78 / max(len(modes_ord), 1)
    pal = {"raw": "#9e9d93", "thermo_qm": "#2a78d6", "full_qm": "#2c9f74"}
    for idx, mode in enumerate(modes_ord):
        sub = bdf[bdf["mode"] == mode].set_index("strategy")
        vals = [sub.loc[s, "winner_time_mae_min"] if s in sub.index else np.nan for s in strats_ord]
        ax.bar(x + (idx - (len(modes_ord) - 1) / 2) * w, vals, width=w, color=pal[mode], label=mode)
    ax.set_xticks(x)
    ax.set_xticklabels(strats_ord, rotation=15, ha="right")
    ax.set_ylabel("Out-of-fold winner time MAE (min)")
    ax.set_title("End-to-end task_v1: raw vs quantile-mapped forecasts")
    ax.legend(frameon=False, fontsize=8.5)
ax.spines[["top", "right"]].set_visible(False)

# Panel B: 4-pillar relative skill breakdown + composite weight
ax = axes[1]
pb = mw.get("pillar_rel_rmse") or {}
gw = mw.get("global") or {}
if gw:
    mids = [m for m, _ in sorted(gw.items(), key=lambda kv: -kv[1])]
    p_df = pd.DataFrame([{"model": m, "comp_weight": gw[m], **(pb.get(m) or {})} for m in mids])
    display(p_df.round(3))
    y = np.arange(len(mids))
    ax.barh(y, [gw[m] * 100 for m in mids], color="#cf5222", alpha=0.88)
    for yi, m in enumerate(mids):
        ax.text(gw[m] * 100 + 0.3, yi, f"{gw[m]:.1%}", va="center", fontsize=8.5)
    ax.set_yticks(y)
    ax.set_yticklabels(mids)
    ax.invert_yaxis()
    ax.set_xlabel("Composite global weight (%)")
    ax.set_title("4-pillar composite global weights (wind + ceiling + climb + task)")
    ax.set_xlim(0, max(gw.values()) * 125)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()